# Generate KEEN subject VP-k representation?

## Create Scalers for VP-k feature
- Grab train subjects data.
- Extract last token hidden state for prompt fed to the model per train subject.
- Project hidden state to vocab space per train subject.
- Fit MinMax scaler on trained logits in vocab space.
- Save fitted scalers as .joblib file on HF

In [ ]:
# @title Imports
from __future__ import annotations

import os
from typing import Callable, Dict, List, Optional, Protocol, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from huggingface_hub import hf_hub_download
from transformers import PreTrainedTokenizerBase

In [ ]:
# @title Setup Minimal model protocol and prompt template
class MTLike(Protocol):
    tokenizer: PreTrainedTokenizerBase
    model: torch.nn.Module
    num_layers: int


def document_prefix(subject: str) -> str:
    return f"This document describes {subject}"

In [ ]:
# @title Get train subject data

from datasets import load_dataset

import pandas as pd
from huggingface_hub import hf_hub_download

train_subjects_df_path = hf_hub_download(
    repo_id="dhgottesman/keen_estimating_knowledge_in_llms",
    filename="popqa_train_subjects.csv",
    repo_type="dataset"
)

train_subjects_df = pd.read_csv(train_subjects_df_path)

# Experiment with 16 data first
# train_subjects_df = train_subjects_df.head(16)

popqa_train_subjects.csv: 0.00B [00:00, ?B/s]

In [ ]:
# @title Extract last-token hidden vectors per layer

@torch.no_grad()
def extract_last_token_hidden_states(
    mt: MTLike,
    df: pd.DataFrame,
    layers: Sequence[int],
    prompt_func: Callable[[str], str],
    batch_size: int = 16,
    device: Optional[torch.device] = None,
) -> Tuple[pd.DataFrame, Dict[int, np.ndarray]]:
    """
    Extracts last-non-pad-token hidden vectors for each layer in `layers`.

    Returns:
      df_used: DataFrame whose rows align with X_by_layer matrices
      X_by_layer: dict[layer_idx] -> np.ndarray of shape (N, d_model)
    """
    if "subject" not in df.columns:
        raise ValueError("Input df must contain a 'subject' column.")

    layers = list(layers)
    if len(layers) == 0:
        raise ValueError("layers must be non-empty.")

    # Choose device
    if device is None:
        try:
            device = next(mt.model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df_used = df.copy()
    df_used = df_used.dropna(subset=["subject"]).reset_index(drop=True)
    df_used["subject"] = df_used["subject"].astype(str)

    tokenizer = mt.tokenizer
    model = mt.model

    N = len(df_used)
    if N == 0:
        raise ValueError("No rows to process after dropping NaN subjects.")

    # Accumulate per layer
    per_layer_rows: Dict[int, List[np.ndarray]] = {layer: [] for layer in layers}

    for start in range(0, N, batch_size):
        batch = df_used.iloc[start : start + batch_size]
        subjects = batch["subject"].tolist()
        prompts = [prompt_func(s) for s in subjects]

        tok = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        out = model(**tok, output_hidden_states=True)

        # Find last non-pad token index for each example
        attn = tok["attention_mask"]                 # (B, S) with 1 for real tokens
        last_pos = (attn.sum(dim=1) - 1).long()      # (B,)

        for b_idx in range(len(subjects)):
            pos = int(last_pos[b_idx].item())
            for layer in layers:
                vec = out.hidden_states[layer][b_idx, pos, :].detach().float().cpu().numpy()
                per_layer_rows[layer].append(vec)

    # Stack into (N, d_model) per layer
    X_by_layer: Dict[int, np.ndarray] = {}
    for layer in layers:
        X_by_layer[layer] = np.stack(per_layer_rows[layer], axis=0).astype(np.float32)

    return df_used, X_by_layer

In [ ]:
# @title Project hidden states into vocab space per subject per layer

@torch.no_grad()
def project_hidden_states_to_vocab_space(
    mt: GPTModelAndTokenizer,
    X_by_layer: Dict[int, np.ndarray],
    layers: Sequence[int],
    batch_size: int = 64,
    device: Optional[torch.device] = None,
    dtype: torch.dtype = torch.float16,
) -> Dict[int, np.ndarray]:
    """
    Returns per layer:
      logits_by_layer: dict[layer_idx] -> np.ndarray of shape (N, vocab)
    """
    if device is None:
        try:
            device = next(mt.model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    layers = list(layers)
    mt.model.eval()

    logits_by_layer: Dict[int, np.ndarray] = {layer: [] for layer in layers}

    for layer in layers:
        X = X_by_layer[layer]  # (N, d_model)
        N, _ = X.shape

        all_vals = []

        for start in range(0, N, batch_size):
            chunk = torch.tensor(X[start:start+batch_size], device=device, dtype=dtype)  # (B, d_model)
            logits = mt.vocabulary_projection_function(chunk, layer)  # (B, vocab)
            all_vals.append(logits.float().detach().cpu().numpy().astype(np.float32))

        logits_by_layer[layer] = np.concatenate(all_vals, axis=0)

    return logits_by_layer

In [ ]:

# @title Setup model and tokenizer
import torch
import transformers
import re

def set_requires_grad(requires_grad, *models):
  for model in models:
    if isinstance(model, torch.nn.Module):
      for param in model.parameters():
        param.requires_grad = requires_grad
    elif isinstance(model, (torch.nn.Parameter, torch.Tensor)):
      model.requires_grad = requires_grad
    else:
      assert False, "unknown type %r" % type(model)


class GPTModelAndTokenizer:
  """An object to hold a GPT-style language model and tokenizer."""

  def __init__(
      self,
      model_name=None,
      model=None,
      tokenizer=None,
      low_cpu_mem_usage=False,
      torch_dtype=None,
      ):
    if tokenizer is None:
      assert model_name is not None
      tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    if model is None:
      assert model_name is not None
      model = transformers.AutoModelForCausalLM.from_pretrained(
          model_name, low_cpu_mem_usage=low_cpu_mem_usage,
          torch_dtype=torch_dtype
          )
      set_requires_grad(False, model)
    self.tokenizer = tokenizer
    self.model = model
    self.layer_names = [
        n
        for n, _ in model.named_modules()
        if (re.match(r"^(transformer|gpt_neox)\.(h|layers)\.\d+$", n))
    ]
    self.num_layers = len(self.layer_names)
    self.vocabulary_projection_function = lambda x, layer: self.model.lm_head(self.model.transformer.ln_f(x)) if layer < self.num_layers else self.model.lm_head(x)
    self.mlp_hidden_size = self.model.config.n_embd * 4
    print(self.mlp_hidden_size)
    print(self.model.config)

  def __repr__(self):
    """String representation of this class.
    """
    return (
        f"ModelAndTokenizer(model: {type(self.model).__name__} "
        f"[{self.num_layers} layers], "
        f"tokenizer: {type(self.tokenizer).__name__})"
        )

In [ ]:
# @title Load gpt2-xl
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mt = GPTModelAndTokenizer(model_name="gpt2-xl", torch_dtype=torch.float16)
mt.tokenizer.pad_token = mt.tokenizer.eos_token # Set pad_token to eos_token
mt.model = mt.model.to(device)
mt.model.eval()

print("num_layers:\n", mt.num_layers)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

6400
GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float16",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}

num_layers:
 48


In [ ]:
num_layers = mt.num_layers
layers = list(range(int(num_layers * 0.75) - 3, int(num_layers * 0.75)))
_, X_by_layer = extract_last_token_hidden_states(
    mt=mt,
    df=train_subjects_df,
    layers=layers,
    prompt_func=document_prefix,
    batch_size=16,
    device=None,
)

logits_by_layer = project_hidden_states_to_vocab_space(
    mt=mt,
    X_by_layer=X_by_layer, # Pass the dictionary, not the tuple
    layers=layers,
    batch_size=16,
    device=None,
    dtype=torch.float16,
)

In [ ]:
# @title Fit MinMaxScaler per layer save to .joblib

from sklearn.preprocessing import MinMaxScaler
import joblib
from datetime import datetime


def fit_minmax_scalers_per_layer(
    logits_by_layer: dict[int, np.ndarray],
    layers: Sequence[int],
) -> list[MinMaxScaler]:
    """
    Fits one MinMaxScaler per layer on X_layer with shape (N_train, d_model).
    Returns scalers in the same order as `layers`.
    """
    scalers: list[MinMaxScaler] = []
    for layer in layers:
        X = logits_by_layer[layer]  # (N_train, d_model)
        scaler = MinMaxScaler()  # feature-wise scaling across samples (TRAIN subjects)
        scaler.partial_fit(X)
        scalers.append(scaler)
    return scalers

# Run minmax scaler per layer
scalers = fit_minmax_scalers_per_layer(logits_by_layer, layers=layers)

artifact = {
    "repo_note": "MinMaxScalers fit on TRAIN subjects only",
    "model_name": "gpt2-xl",
    "layers": list(layers),
    "d_vocab": mt.model.config.vocab_size,
    "prompt_template": "This document describes {subject}",
    "scalers": scalers,
}

local_path = f"gpt2xl_keen_minmax_scalers_logits.joblib"
joblib.dump(artifact, local_path)

print("Saved:", local_path)

Saved: gpt2xl_keen_minmax_scalers_logits.joblib


In [ ]:
layer_idx = layers[0]
X_by_layer[layer_idx].shape, logits_by_layer[layer_idx].shape

((2224, 1600), (2224, 50257))

In [ ]:
# @title Setup HF token as env var
import os
from google.colab import userdata

# Get the token from Colab secrets
colab_hf_token = userdata.get("HF_TOKEN")

# Set it as an environment variable for the Python process
if colab_hf_token:
    os.environ["HF_TOKEN"] = colab_hf_token
    print("HF_TOKEN successfully set as environment variable.")
else:
    raise RuntimeError("HF_TOKEN not found in Colab secrets. Please add it before running this cell.")

HF_TOKEN successfully set as environment variable.


In [ ]:
import os
from huggingface_hub import HfApi
from google.colab import userdata

# Retrieve HF_TOKEN from Colab secrets
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found in env. Use export HF_TOKEN=.... Please add it.")

repo_id = "kokolamba/keen_popqa_gpt2xl_generations"

api = HfApi(token=HF_TOKEN)

path_in_repo = f"scalers/{os.path.basename(local_path)}"  # store under a scalers/ folder

api.upload_file(
    path_or_fileobj=local_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="dataset",
    commit_message=f"Add per-layer MinMaxScalers for logits (train-set-only) for GPT-2 XL: layers={layers}",
)

print("Uploaded to:", f"{repo_id}/{path_in_repo}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...max_scalers_logits.joblib:  27%|##6       |  807kB / 3.02MB            

Uploaded to: kokolamba/keen_popqa_gpt2xl_generations/scalers/gpt2xl_keen_minmax_scalers_logits.joblib


## Create and save datasets for logits for all subjects data

- Use saved scalers to fit all subjects data(train, val and test).
- Average normalized logits across layer for each subject.
- Train probe with the averaged logits.
- Extract `top-k` token ids on the learned weights of the probe.
- Use the extracted `top-k` token ids to extract logits per layer per subject.
- Normalize and average and use it to train the probe.
- Hopefully, this generates good result.

In [ ]:
# @title Download and Install dependencies
!pip install -qq --progress-bar off \
  torch torchvision torchaudio \
  transformers datasets \
  numpy pandas scipy scikit-learn \
  tqdm einops \
  wandb \
  matplotlib plotly

In [ ]:
# @title Imports
from __future__ import annotations

import os
from typing import Callable, Dict, List, Optional, Protocol, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler
from huggingface_hub import hf_hub_download
from transformers import PreTrainedTokenizerBase

In [ ]:
# @title Generate qa accuracy for all subjects
from datasets import load_dataset
from ast import literal_eval
import pandas as pd


def qa_accuracy(model_name):
    def _generation(model_name, question, generation):
        if model_name in ["llama2_7B", "llama2_13B", "vicuna_7B", "vicuna_13B"]:
            generation = generation.replace(f"<s> {question}", "").strip()
            generation = generation.replace(f"</s>", "").strip()
        elif model_name in ["gpt2_xl", "pythia_6B", "pythia_12B"]:
            generation = generation.replace(f"{question}", "").strip()
        return generation

    df = load_dataset(f"kokolamba/keen_popqa_gpt2xl_generations")
    df = df["train"].to_pandas()
    df["deterministic_generation"] = df.apply(lambda row: _generation(model_name, row["question"], row["deterministic_generation"]), axis=1)

    questions = load_dataset("dhgottesman/keen_estimating_knowledge_in_llms", data_files="popqa_questions.csv")
    questions = questions["train"].to_pandas()
    questions["possible_answers"] = questions["possible_answers"].apply(lambda x: literal_eval(x))
    questions = questions.rename(columns={"subj": "subject"})

    df = df.merge(questions, on="question").dropna()

    def label_generation(generation, answers):
        for answer in answers:
            if answer.lower() in generation.lower():
                return 3
        for hedged_answer in ["nobody knows", "I'm sorry", "I can't seem to find the answer", "you help me", "anyone help me", "I'm not sure", "I don't know"]:
            if hedged_answer.lower() in generation.lower():
                return 2
        # if generation is an empty, this is taking to be a hedged answer
        if generation == "":
                return 2
        return 1

    def binary_label(label, class_label):
        return 1 if label == class_label else 0

    df["generation_label"] = df.apply(lambda row: label_generation(row["deterministic_generation"], row["possible_answers"]), axis=1)
    # Multiple answers for each question, if one of them is correct then mark the question as correct.
    idx = df.groupby(['subject', 's_uri', 'prop'])["generation_label"].idxmax()
    df = df.loc[idx]

    # We want to compute correct, hedged, mistake accuracy.
    df["correct"] = df["generation_label"].apply(lambda x: binary_label(x, 3))
    df["hedge"] = df["generation_label"].apply(lambda x: binary_label(x, 2))
    df["mistake"] = df["generation_label"].apply(lambda x: binary_label(x, 1))

    result_df = df.groupby(['subject', 's_uri', "label"]).agg(
        total_examples=('generation_label', 'count'),
        accuracy=('correct', 'mean')
        # hedged_frac=('hedge', 'mean'),
        # mistake_frac=('mistake', 'mean')
    ).reset_index()

    result_df = result_df[result_df["total_examples"] > 1]
    return result_df # ["subject", "accuracy", "total_examples"]

In [ ]:
# Run qa_accuracy to generate qa accuracy data
model_name = "gpt2_xl"
qa_accuracy_df = qa_accuracy(model_name)

README.md:   0%|          | 0.00/332 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.02M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19203 [00:00<?, ? examples/s]

popqa_questions.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# @title Load scalers artifact from your HF dataset repo

def load_scalers_artifact_from_hub(
    repo_id: str,
    path_in_repo: str,
) -> dict:
    """
    Downloads a .joblib artifact from a HF dataset repo and loads it.
    Expects artifact to contain:
      - "layers": list[int]
      - "scalers": list[MinMaxScaler] aligned with layers
    """
    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=path_in_repo,
    )
    artifact = joblib.load(local_path)

    if "layers" not in artifact or "scalers" not in artifact:
        raise ValueError("Artifact must contain keys 'layers' and 'scalers'.")

    layers = artifact["layers"]
    scalers = artifact["scalers"]
    if len(layers) != len(scalers):
        raise ValueError("artifact['layers'] and artifact['scalers'] must have same length.")

    # Basic validation: ensure each is a fitted MinMaxScaler
    for s in scalers:
        if not isinstance(s, MinMaxScaler):
            raise TypeError(f"Expected MinMaxScaler; got {type(s)}")

    return artifact


repo_id = "kokolamba/keen_popqa_gpt2xl_generations"
scalers_path_in_repo = f"scalers/gpt2xl_keen_minmax_scalers_logits.joblib"

artifact = load_scalers_artifact_from_hub(repo_id, scalers_path_in_repo)
layers: List[int] = list(artifact["layers"])
scalers: List[MinMaxScaler] = list(artifact["scalers"])

print("Loaded layers:", layers)
print("Num scalers:", len(scalers))

scalers/gpt2xl_keen_minmax_scalers_logit(…):   0%|          | 0.00/3.02M [00:00<?, ?B/s]

Loaded layers: [33, 34, 35]
Num scalers: 3


In [ ]:
# @title Extract last-token hidden vectors per layer

@torch.no_grad()
def extract_last_token_hidden_states(
    mt: MTLike,
    df: pd.DataFrame,
    layers: Sequence[int],
    prompt_func: Callable[[str], str],
    batch_size: int = 16,
    device: Optional[torch.device] = None,
) -> Tuple[pd.DataFrame, Dict[int, np.ndarray]]:
    """
    Extracts last-non-pad-token hidden vectors for each layer in `layers`.

    Returns:
      df_used: DataFrame whose rows align with X_by_layer matrices
      X_by_layer: dict[layer_idx] -> np.ndarray of shape (N, d_model)
    """
    if "subject" not in df.columns:
        raise ValueError("Input df must contain a 'subject' column.")

    layers = list(layers)
    if len(layers) == 0:
        raise ValueError("layers must be non-empty.")

    # Choose device
    if device is None:
        try:
            device = next(mt.model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df_used = df.copy()
    df_used = df_used.dropna(subset=["subject"]).reset_index(drop=True)
    df_used["subject"] = df_used["subject"].astype(str)

    tokenizer = mt.tokenizer
    model = mt.model

    N = len(df_used)
    if N == 0:
        raise ValueError("No rows to process after dropping NaN subjects.")

    # Accumulate per layer
    per_layer_rows: Dict[int, List[np.ndarray]] = {layer: [] for layer in layers}

    for start in range(0, N, batch_size):
        batch = df_used.iloc[start : start + batch_size]
        subjects = batch["subject"].tolist()
        prompts = [prompt_func(s) for s in subjects]

        tok = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        out = model(**tok, output_hidden_states=True)

        # Find last non-pad token index for each example
        attn = tok["attention_mask"]                 # (B, S) with 1 for real tokens
        last_pos = (attn.sum(dim=1) - 1).long()      # (B,)

        for b_idx in range(len(subjects)):
            pos = int(last_pos[b_idx].item())
            for layer in layers:
                vec = out.hidden_states[layer][b_idx, pos, :].detach().float().cpu().numpy()
                per_layer_rows[layer].append(vec)

    # Stack into (N, d_model) per layer
    X_by_layer: Dict[int, np.ndarray] = {}
    for layer in layers:
        X_by_layer[layer] = np.stack(per_layer_rows[layer], axis=0).astype(np.float32)

    return df_used, X_by_layer

In [ ]:
# @title Project hidden states into vocab space per subject per layer

@torch.no_grad()
def project_hidden_states_to_vocab_space(
    mt: GPTModelAndTokenizer,
    X_by_layer: Dict[int, np.ndarray],
    layers: Sequence[int],
    batch_size: int = 64,
    device: Optional[torch.device] = None,
    dtype: torch.dtype = torch.float16,
) -> Dict[int, np.ndarray]:
    """
    Returns per layer:
      logits_by_layer: dict[layer_idx] -> np.ndarray of shape (N, vocab)
    """
    if device is None:
        try:
            device = next(mt.model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    layers = list(layers)
    mt.model.eval()

    logits_by_layer: Dict[int, np.ndarray] = {layer: [] for layer in layers}

    for layer in layers:
        X = X_by_layer[layer]  # (N, d_model)
        N, _ = X.shape

        all_vals = []

        for start in range(0, N, batch_size):
            chunk = torch.tensor(X[start:start+batch_size], device=device, dtype=dtype)  # (B, d_model)
            logits = mt.vocabulary_projection_function(chunk, layer)  # (B, vocab)
            all_vals.append(logits.float().detach().cpu().numpy().astype(np.float32))

        logits_by_layer[layer] = np.concatenate(all_vals, axis=0)

    return logits_by_layer

In [ ]:
# @title Transform logits with saved scalers

def transform_logits_with_scalers(
    logits_by_layer: Dict[int, np.ndarray],
    layers: Sequence[int],
    scalers: Sequence[MinMaxScaler],
) -> Tuple[List[List[List[float]]], List[List[float]]]:
    """
    Applies scaler[i] to logit_by_layer[layers[i]].

    Returns:
      scaled_layers_per_row:
        list length N, each item is list length L of vectors (list[float]) (d_vocab,)
        i.e. scaled_layers_per_row[row][layer_i][dim]
      avg_per_row:
        list length N, each item is averaged vector across layers (list[float]) (d_vocab,)
    """
    layers = list(layers)
    scalers = list(scalers)
    if len(layers) != len(scalers):
        raise ValueError("layers and scalers must have the same length.")

    # Transform each layer matrix: (N, d_model)
    scaled_mats: List[np.ndarray] = []
    for i, layer in enumerate(layers):
        X = logits_by_layer[layer]
        Xs = scalers[i].transform(X)  # feature-wise scaling across samples, using train-fitted min/max
        scaled_mats.append(Xs.astype(np.float32))

    # Stack -> (L, N, d_vocab) then transpose -> (N, L, d_vocab)
    stacked = np.stack(scaled_mats, axis=0).transpose(1, 0, 2)  # (N, L, d)

    # Average across layers -> (N, d_model)
    avg = stacked.mean(axis=1)  # (N, d)

    # Convert to python lists (parquet-friendly)
    scaled_layers_per_row: List[List[List[float]]] = [
        [stacked[r, li, :].tolist() for li in range(stacked.shape[1])]
        for r in range(stacked.shape[0])
    ]
    avg_per_row: List[List[float]] = [avg[r, :].tolist() for r in range(avg.shape[0])]

    return scaled_layers_per_row, avg_per_row


def attach_scaled_columns_to_df(
    df_original: pd.DataFrame,
    df_features_indexed: pd.DataFrame,
    scaled_layers_per_row: List[List[List[float]]],
    avg_per_row: List[List[float]],
) -> pd.DataFrame:
    """
    Attaches columns layer_0..layer_{L-1} and hidden_states to df_original by matching on 'subject'.

    - df_features_indexed must align row-for-row with scaled_layers_per_row / avg_per_row
      (it is the df_used returned from extraction)
    - If df_original contains duplicate subjects, values are broadcast via merge.
    """
    L = len(scaled_layers_per_row[0]) if scaled_layers_per_row else 0
    layer_cols = [f"layer_{i}" for i in range(L)]

    # Ensure df_features_indexed contains the unique key columns needed for merging
    merge_cols = ["subject", "s_uri", "label"]
    df_feat = df_features_indexed[merge_cols].copy()
    for i, col in enumerate(layer_cols):
        df_feat[col] = [scaled_layers_per_row[r][i] for r in range(len(df_feat))]
    df_feat["logits"] = avg_per_row

    # Merge back into original df (retains all original columns) using the unique key
    df_out = df_original.copy()
    df_out["subject"] = df_out["subject"].astype(str)
    df_out = df_out.merge(df_feat, on=merge_cols, how="left")

    return df_out

In [ ]:

# @title Setup model and tokenizer
import torch
import transformers
import re

def set_requires_grad(requires_grad, *models):
  for model in models:
    if isinstance(model, torch.nn.Module):
      for param in model.parameters():
        param.requires_grad = requires_grad
    elif isinstance(model, (torch.nn.Parameter, torch.Tensor)):
      model.requires_grad = requires_grad
    else:
      assert False, "unknown type %r" % type(model)


class GPTModelAndTokenizer:
  """An object to hold a GPT-style language model and tokenizer."""

  def __init__(
      self,
      model_name=None,
      model=None,
      tokenizer=None,
      low_cpu_mem_usage=False,
      torch_dtype=None,
      ):
    if tokenizer is None:
      assert model_name is not None
      tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    if model is None:
      assert model_name is not None
      model = transformers.AutoModelForCausalLM.from_pretrained(
          model_name, low_cpu_mem_usage=low_cpu_mem_usage,
          torch_dtype=torch_dtype
          )
      set_requires_grad(False, model)
    self.tokenizer = tokenizer
    self.model = model
    self.layer_names = [
        n
        for n, _ in model.named_modules()
        if (re.match(r"^(transformer|gpt_neox)\.(h|layers)\.\d+$", n))
    ]
    self.num_layers = len(self.layer_names)
    self.vocabulary_projection_function = lambda x, layer: self.model.lm_head(self.model.transformer.ln_f(x)) if layer < self.num_layers else self.model.lm_head(x)
    self.mlp_hidden_size = self.model.config.n_embd * 4
    print(self.mlp_hidden_size)
    print(self.model.config)

  def __repr__(self):
    """String representation of this class.
    """
    return (
        f"ModelAndTokenizer(model: {type(self.model).__name__} "
        f"[{self.num_layers} layers], "
        f"tokenizer: {type(self.tokenizer).__name__})"
        )

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mt = GPTModelAndTokenizer(model_name="gpt2-xl", torch_dtype=torch.float16)
mt.tokenizer.pad_token = mt.tokenizer.eos_token # Set pad_token to eos_token
mt.model = mt.model.to(device)
mt.model.eval()

print("num_layers:\n", mt.num_layers)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

6400
GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float16",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}

num_layers:
 48


In [ ]:
# @title Setup Minimal model protocol and prompt template
class MTLike(Protocol):
    tokenizer: PreTrainedTokenizerBase
    model: torch.nn.Module
    num_layers: int


def document_prefix(subject: str) -> str:
    return f"This document describes {subject}"

In [ ]:
# @title Overall function that does the whole job

def build_scaled_logits_features_for_df(
    mt: MTLike,
    df_split: pd.DataFrame,
    layers: Sequence[int],
    scalers: Sequence[MinMaxScaler],
    prompt_func: Callable[[str], str] = document_prefix,
    batch_size: int = 16,
    device: Optional[torch.device] = None,
) -> pd.DataFrame:
    """
    End-to-end:
      1) Extract raw hidden state for `layers` (last token).
      2) Project hidden states into vocab space.
      2) Scale each layer with provided train-fitted scalers.
      3) Add layer_0..layer_{L-1} and hidden_states columns.
      4) Return df with original columns retained.
    """
    df_used, X_by_layer = extract_last_token_hidden_states(
        mt=mt,
        df=df_split,
        layers=layers,
        prompt_func=prompt_func,
        batch_size=batch_size,
        device=device,
    )

    logits_by_layer = project_hidden_states_to_vocab_space(
        mt=mt,
        X_by_layer=X_by_layer,
        layers=layers,
        batch_size=batch_size,
        device=device,
        dtype=torch.float16,
    )

    scaled_layers_per_row, avg_per_row = transform_logits_with_scalers(
        logits_by_layer=logits_by_layer,
        layers=layers,
        scalers=scalers,
    )

    df_out = attach_scaled_columns_to_df(
        df_original=df_split,
        df_features_indexed=df_used,
        scaled_layers_per_row=scaled_layers_per_row,
        avg_per_row=avg_per_row,
    )

    return df_out

In [ ]:

# @title Build full logits features
full_df = build_scaled_logits_features_for_df(
    mt=mt,
    df_split=qa_accuracy_df,
    layers=layers,
    scalers=scalers,
    prompt_func=document_prefix,
    batch_size=1,
    device=None,
)

In [ ]:
# @title Setup HF_TOKEN properly in colab
# import os
# from google.colab import userdata

# # Get the token from Colab secrets
# colab_hf_token = userdata.get("HF_TOKEN")

# # Set it as an environment variable for the Python process
# if colab_hf_token:
#     os.environ["HF_TOKEN"] = colab_hf_token
#     print("HF_TOKEN successfully set as environment variable.")
# else:
#     raise RuntimeError("HF_TOKEN not found in Colab secrets. Please add it before running this cell.")

# Enter huggngface token via the CLI
!huggingface-cli login

In [ ]:
# @title Save dataframe as parquet upload to huggingface dataset repo

from __future__ import annotations

import os
from typing import Optional

import pandas as pd
from huggingface_hub import HfApi


def save_df_to_parquet_and_upload(
    df: pd.DataFrame,
    *,
    local_path: str,
    repo_id: str,
    path_in_repo: str,
    token: Optional[str] = None,
    repo_type: str = "dataset",
    commit_message: Optional[str] = None,
    parquet_engine: str = "pyarrow",
    compression: str = "snappy",
    index: bool = False,
) -> str:
    """
    Save a DataFrame to a Parquet file and upload it to a Hugging Face *dataset* repo.

    Args:
        df: DataFrame to save.
        local_path: Local filepath to write, e.g. "train_features.parquet".
        repo_id: HF repo id, e.g. "kokolamba/keen_popqa_gpt2xl_generations".
        path_in_repo: Destination path inside the repo, e.g. "features/train_features.parquet".
        token: HF token with write access. If None, uses env var HF_TOKEN.
        repo_type: Should be "dataset" for dataset repos.
        commit_message: Commit message for upload. If None, a default is used.
        parquet_engine: Parquet engine, usually "pyarrow".
        compression: Parquet compression codec, e.g. "snappy", "zstd", or None.
        index: Whether to store the DataFrame index in Parquet.

    Returns:
        The uploaded path reference: f"{repo_id}/{path_in_repo}"
    """
    # 1) Save parquet locally
    df.to_parquet(
        local_path,
        engine=parquet_engine,
        compression=compression,
        index=index,
    )

    # 2) Upload to HF
    hf_token = token or os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF token not provided. Pass token=... or set HF_TOKEN env var.")

    api = HfApi(token=hf_token)

    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=repo_id,
        repo_type=repo_type,
        commit_message=commit_message or f"Upload {path_in_repo}",
    )

    return f"{repo_id}/{path_in_repo}"

`torch_dtype` is deprecated! Use `dtype` instead!


6400
GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float16",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 1600,
  "n_head": 25,
  "n_inner": null,
  "n_layer": 48,
  "n_positions": 1024,
  "output_past": true,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.3",
  "use_cache": true,
  "vocab_size": 50257
}

num_layers:
 48


In [ ]:
# Save to local path and upload to huggingface repo

repo_id = "kokolamba/keen_popqa_gpt2xl_generations"
path_in_repo = "features/logits.parquet"
local_path = "logits.parquet"

save_df_to_parquet_and_upload(
    df=full_df,
    local_path=local_path,
    repo_id=repo_id,
    path_in_repo=path_in_repo,
    commit_message=f"Upload qa accuracy data with avg and layer logits"
)

In [ ]:
# Let load in logits data from huggingface repo

import pandas as pd
from huggingface_hub import hf_hub_download

repo_id = "kokolamba/keen_popqa_gpt2xl_generations"
path_in_repo = "features/logits.parquet"

local_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=path_in_repo,
)

df = pd.read_parquet(local_path)
df.head()

features/top_200_logits.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

,subject,s_uri,label,total_examples,accuracy,layer_0,layer_1,layer_2,top_200_logits
0,'71,http://www.wikidata.org/entity/Q12100227,head,7,0.142857,"[0.31902259588241577, 0.251968115568161, 0.390...","[0.35611772537231445, 0.267502099275589, 0.366...","[0.31481584906578064, 0.25293925404548645, 0.3...","[0.3299853801727295, 0.2574698030948639, 0.361..."
1,(I Can't Get No) Satisfaction,http://www.wikidata.org/entity/Q158553,head,5,0.000000,"[0.16516464948654175, 0.20569351315498352, 0.2...","[0.14090675115585327, 0.171124666929245, 0.256...","[0.14184874296188354, 0.18060138821601868, 0.2...","[0.14930671453475952, 0.1858065128326416, 0.25..."
2,10,http://www.wikidata.org/entity/Q184591,head,7,0.000000,"[0.31398719549179077, 0.3729940354824066, 0.52...","[0.38176465034484863, 0.4657251536846161, 0.57...","[0.42713215947151184, 0.5158596038818359, 0.62...","[0.3742946684360504, 0.4515262544155121, 0.577..."
3,10 Years,http://www.wikidata.org/entity/Q2579741,head,7,0.285714,"[0.5870150327682495, 0.6758823394775391, 0.624...","[0.628197968006134, 0.7179470062255859, 0.6463...","[0.6085230112075806, 0.7175709009170532, 0.585...","[0.607912003993988, 0.7038000226020813, 0.6190..."
4,13,http://www.wikidata.org/entity/Q3018412,head,5,0.200000,"[0.322379469871521, 0.3373982012271881, 0.4269...","[0.3633657693862915, 0.4329158365726471, 0.476...","[0.4018609821796417, 0.49151507019996643, 0.49...","[0.36253538727760315, 0.420609712600708, 0.464..."


In [ ]:
len(df), df.columns, len(df.top_200_logits[0])

(3491,
 Index(['subject', 's_uri', 'label', 'total_examples', 'accuracy', 'layer_0',
        'layer_1', 'layer_2', 'top_200_logits'],
       dtype='object'),
 200)

In [ ]:
# @title Delete .joblib file from huggingface dataset repo

# from huggingface_hub import HfApi
# import os

# HF_TOKEN = os.environ["HF_TOKEN"]  # must have write access
# api = HfApi(token=HF_TOKEN)

# if not HF_TOKEN:
#     raise RuntimeError("HF_TOKEN not found in env. Use export HF_TOKEN=.... Please add it.")

# repo_id = "kokolamba/keen_popqa_gpt2xl_generations"
# path_in_repo = "features/residual_stream.parquet"

# api.delete_file(
#     repo_id=repo_id,
#     path_in_repo=path_in_repo,
#     repo_type="dataset",
#     commit_message=f"Delete {path_in_repo}",
# )

# print("Deleted:", f"{repo_id}/{path_in_repo}")